# Planet Hunters folded-view CNN

This notebook trains the raw-cadence plus phase-folded CNN on a Colab GPU, checks camera-grouped macro-F1, and writes a separate blend candidate only if it clears the validation gate. It does not overwrite the existing α=0.40 submission.

Copy these items into one Drive folder named `planet-hunters-round1`: `planet_hunters/`, `cache/`, `train_stars.csv`, `test_stars.csv`, `sample_submission.csv`, and `artifacts_cnn/lightgbm_ensemble.joblib`, `artifacts_cnn/camera_grouped_probabilities.npy`. The cache folder needs both train/test `*_cnn_sequences.npy`, `*_folded_sequences.npy`, and `*_features.pkl`. No raw flux/centroid arrays are needed if those caches are present (about 1.2 GB total).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys
ROOT = Path('/content/drive/MyDrive/planet-hunters-round1')
assert (ROOT / 'planet_hunters' / 'folded_cnn.py').exists(), f'Missing project files under {ROOT}'
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print('Project:', ROOT)

In [ ]:
# Colab normally includes PyTorch, NumPy, pandas, and scikit-learn.
import torch, numpy as np, pandas as pd
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), 'In Colab choose Runtime > Change runtime type > GPU.'
!pip -q install lightgbm
required = [
    'cache/train_cnn_sequences.npy', 'cache/test_cnn_sequences.npy',
    'cache/train_folded_sequences.npy', 'cache/test_folded_sequences.npy',
    'cache/train_features.pkl', 'cache/test_features.pkl',
    'artifacts_cnn/lightgbm_ensemble.joblib',
    'artifacts_cnn/camera_grouped_probabilities.npy',
    'train_stars.csv', 'test_stars.csv', 'sample_submission.csv'
]
missing = [p for p in required if not (ROOT / p).exists()]
assert not missing, f'Missing files: {missing}'
print('All cached inputs are present.')

In [ ]:
# Copy data to the Colab VM's local disk for much faster random batch reads.
import shutil
LOCAL = Path('/content/planet-hunters-round1')
if LOCAL.exists(): shutil.rmtree(LOCAL)
shutil.copytree(ROOT, LOCAL, ignore=shutil.ignore_patterns('*.npy', '*.pkl', '*.pt', '*.joblib', '*.zip', 'TimesFM_Google', '.git', 'artifacts', 'artifacts_cnn', 'artifacts_folded_cnn'))
for rel in required:
    dest = LOCAL / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ROOT / rel, dest)
sys.path.insert(0, str(LOCAL))
os.chdir(LOCAL)
print('Local working copy:', LOCAL)

In [ ]:
# Four held-out camera folds, five-epoch cap. Partial checkpoints are written after every fold.
import subprocess
cmd = [sys.executable, '-u', '-m', 'planet_hunters.folded_cnn',
       '--data-dir', str(LOCAL), '--cache-dir', str(LOCAL / 'cache'),
       '--output-dir', str(LOCAL / 'artifacts_folded_cnn'), '--epochs', '5', '--batch-size', '128']
subprocess.run(cmd, cwd=LOCAL, check=True)

In [ ]:
# Nested camera-fold alpha selection and a separate submission if the CNN blend passes the gate.
import json, joblib
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
from planet_hunters.modeling import LABELS, _global_probabilities, make_submission
out = LOCAL / 'artifacts_folded_cnn'
stars = pd.read_csv(LOCAL / 'train_stars.csv')
y, camera = stars['label'].to_numpy(dtype=str), stars['camera'].to_numpy()
lgb_oof = np.load(LOCAL / 'artifacts_cnn/camera_grouped_probabilities.npy')
cnn_oof = np.load(out / 'camera_grouped_probabilities.npy')
def macro(y0, p): return f1_score(y0, np.asarray(LABELS)[p.argmax(axis=1)], labels=LABELS, average='macro', zero_division=0)
splits = list(GroupKFold(n_splits=4).split(np.zeros(len(y)), y, groups=camera))
alphas = np.arange(0, .501, .05)
fold_scores, selected = [], []
for tr, va in splits:
    a = float(alphas[np.argmax([macro(y[tr], (1-x)*lgb_oof[tr] + x*cnn_oof[tr]) for x in alphas])])
    selected.append(a)
    fold_scores.append(macro(y[va], (1-a)*lgb_oof[va] + a*cnn_oof[va]))
baseline = [macro(y[va], lgb_oof[va]) for _, va in splits]
gain = float(np.mean(fold_scores) - np.mean(baseline))
print('Nested alpha per camera:', selected)
print('Folded-CNN blend folds:', np.round(fold_scores, 5).tolist())
print('LGBM baseline folds:', np.round(baseline, 5).tolist())
print(f'Nested camera macro-F1: {np.mean(fold_scores):.5f}; gain: {gain:+.5f}')
accepted = gain >= .003 and all(a >= b - .02 for a,b in zip(fold_scores, baseline))
print('Grouped gate:', accepted)
if accepted:
    alpha = float(np.median(selected))
    features = pd.read_pickle(LOCAL / 'cache/test_features.pkl').drop(columns='id').astype(np.float32)
    ensemble = joblib.load(LOCAL / 'artifacts_cnn/lightgbm_ensemble.joblib')
    assert len(ensemble) % 6 == 0
    n_seeds = len(ensemble) // 6
    lgb_test = np.zeros((7000, len(LABELS)), dtype=np.float32)
    for s in range(n_seeds):
        models = ensemble[s*6:(s+1)*6]
        folds = np.mean([_global_probabilities(m, features) for m in models[:5]], axis=0)
        lgb_test += .5 * (folds + _global_probabilities(models[5], features)) / n_seeds
    cnn_test = np.load(out / 'test_probabilities.npy')
    probs = (1-alpha)*lgb_test + alpha*cnn_test
    probs /= np.maximum(probs.sum(axis=1, keepdims=True), 1e-12)
    test_stars = pd.read_csv(LOCAL / 'test_stars.csv')
    template = pd.read_csv(LOCAL / 'sample_submission.csv')
    sub = make_submission(test_stars, np.asarray(LABELS)[probs.argmax(axis=1)], template, out / f'submission_folded_alpha_{alpha:.2f}.csv')
    print('Created:', out / f'submission_folded_alpha_{alpha:.2f}.csv')
else:
    print('Keep the existing α=0.40 candidate; this folded CNN did not clear the gate.')
shutil.copytree(out, ROOT / 'artifacts_folded_cnn_colab', dirs_exist_ok=True)
print('Results copied to Drive:', ROOT / 'artifacts_folded_cnn_colab')